In [ ]:
# Example 5: Production Patterns and Advanced Features
from einops import rearrange, reduce, repeat
import numpy as np

print("Production patterns and advanced features:")
print("=" * 43)

# Production-grade tensor processing pipeline
batch_data = np.random.randn(8, 224, 224, 3)  # Standard ImageNet batch
print(f"Input batch: {batch_data.shape} (ImageNet format)")

# Advanced Vision Transformer preprocessing
vit_patches = rearrange(batch_data, 'b (h p1) (w p2) c -> b (h w) (p1 p2 c)', p1=16, p2=16)
print(f"ViT patches: {vit_patches.shape} ({vit_patches.shape[1]} patches of {vit_patches.shape[2]} features)")

# Multi-scale feature extraction
print(f"\nMulti-scale feature pyramid:")
scales = [4, 8, 16]
feature_pyramid = {}
for scale in scales:
    # Extract features at different scales
    # Each scale reduces spatial dimensions and increases channel dimension
    features = rearrange(batch_data, 'b (h p1) (w p2) c -> b h w (c p1 p2)', p1=scale, p2=scale)
    feature_pyramid[f'scale_{scale}'] = features
    print(f"  Scale {scale}x{scale}: {batch_data.shape} -> {features.shape}")
    print(f"    Spatial: {224//scale}x{224//scale}, Channels: {3*scale*scale}")

# Attention mechanism simulation
print(f"\nAttention mechanism:")
sequence_len = 196  # 14x14 patches
attention_weights = np.random.rand(8, 12, sequence_len, sequence_len)
attention_weights = attention_weights / attention_weights.sum(axis=-1, keepdims=True)  # Softmax-like normalization
attention_reshaped = rearrange(attention_weights, 'b heads seq1 seq2 -> b (heads seq1) seq2')
print(f"  Attention patterns: {attention_weights.shape} -> {attention_reshaped.shape}")

# Production validation and monitoring
print(f"\nData validation and monitoring:")
data_stats = {
    'mean': reduce(batch_data, 'b h w c -> c', 'mean'),
    'std': np.std(rearrange(batch_data, 'b h w c -> (b h w) c'), axis=0),
    'max': reduce(batch_data, 'b h w c -> c', 'max')
}
print(f"  Channel statistics:")
print(f"    Mean range: [{data_stats['mean'].min():.3f}, {data_stats['mean'].max():.3f}]")
print(f"    Std range: [{data_stats['std'].min():.3f}, {data_stats['std'].max():.3f}]")
print(f"    Max range: [{data_stats['max'].min():.3f}, {data_stats['max'].max():.3f}]")

# Layer-like operations using functional API
print(f"\nLayer-like transformations (functional API):")

# Simulate a learnable Rearrange layer
def rearrange_layer(x, pattern_in, pattern_out, **axes_lengths):
    """Simulate a rearrange layer operation"""
    return rearrange(x, f'{pattern_in} -> {pattern_out}', **axes_lengths)

# Simulate a Reduce layer
def reduce_layer(x, pattern_in, pattern_out, reduction='mean'):
    """Simulate a reduce layer operation"""
    return reduce(x, f'{pattern_in} -> {pattern_out}', reduction)

# Apply layer-like operations
layer_input = np.random.randn(4, 32, 32, 64)
print(f"  Layer input: {layer_input.shape}")

# Rearrange layer: space-to-depth transformation
layer_output1 = rearrange_layer(layer_input, 'b (h h2) (w w2) c', 'b h w (c h2 w2)', h2=2, w2=2)
print(f"  Space-to-depth: {layer_input.shape} -> {layer_output1.shape}")

# Reduce layer: global average pooling
layer_output2 = reduce_layer(layer_input, 'b h w c', 'b c', 'mean')
print(f"  Global avg pool: {layer_input.shape} -> {layer_output2.shape}")

# Depthwise separable convolution simulation
print(f"\nDepthwise separable operations:")
depthwise_input = np.random.randn(2, 64, 64, 32)
print(f"  Input: {depthwise_input.shape}")

# Depthwise: process each channel independently
depthwise_groups = rearrange(depthwise_input, 'b h w c -> b h w c 1')
print(f"  Depthwise grouped: {depthwise_groups.shape}")

# Pointwise: 1x1 convolution across channels
pointwise_ready = rearrange(depthwise_input, 'b h w c -> b (h w) c')
print(f"  Pointwise ready: {pointwise_ready.shape}")

print("\n✓ Production tensor processing pipeline completed successfully!")

In [ ]:
# Example 4: Multi-framework Compatibility and Advanced Patterns
from einops import rearrange, reduce, repeat
import numpy as np

print("Multi-framework compatibility and advanced patterns:")
print("=" * 52)

# Cross-framework tensor operations
np_tensor = np.random.randn(4, 32, 32, 3)  # NHWC format
print(f"NumPy tensor (NHWC): {np_tensor.shape}")

# Framework-agnostic transformations
chw_format = rearrange(np_tensor, 'n h w c -> n c h w')  # Convert to NCHW
print(f"Converted to NCHW: {chw_format.shape}")

# Computer vision preprocessing pipeline
print("\nComputer vision preprocessing:")
feature_maps = np.random.randn(2, 128, 16, 16)  # Deep feature maps (NCHW format)

# Global average pooling
gap_features = reduce(feature_maps, 'batch channels height width -> batch channels', 'mean')
print(f"Global average pooling: {feature_maps.shape} -> {gap_features.shape}")

# Spatial pyramid pooling
print("\nSpatial Pyramid Pooling:")
spp_levels = [1, 2, 4]
spp_features = []
for level in spp_levels:
    # Use proper variable names instead of f-string with literals
    pooled = rearrange(feature_maps, 'b c (h p1) (w p2) -> b c h w (p1 p2)', p1=level, p2=level)
    pooled = reduce(pooled, 'b c h w pool_size -> b (c pool_size)', 'mean')
    spp_features.append(pooled)
    print(f"  SPP level {level}x{level}: pooled to {pooled.shape}")

# Alternative SPP using reduce directly
print("\nAlternative SPP (using reduce directly):")
for level in spp_levels:
    # Reduce spatial dimensions directly
    pooled_alt = reduce(feature_maps, 'b c (h p1) (w p2) -> b (c p1 p2)', 'mean', p1=level, p2=level)
    print(f"  SPP level {level}x{level}: {feature_maps.shape} -> {pooled_alt.shape}")

# NLP attention mechanisms
print("\nNLP attention mechanisms:")
sequence = np.random.randn(2, 64, 768)  # batch, sequence, embedding
multi_head = rearrange(sequence, 'b s (heads dim) -> b heads s dim', heads=12)
print(f"Multi-head format: {sequence.shape} -> {multi_head.shape}")

# Attention pattern analysis
attention_scores = np.random.rand(2, 12, 64, 64)
attention_scores = attention_scores / attention_scores.sum(axis=-1, keepdims=True)
attention_entropy = -np.sum(attention_scores * np.log(attention_scores + 1e-10), axis=-1)
entropy_stats = reduce(attention_entropy, 'batch heads sequence -> heads', 'mean')
print(f"Attention entropy per head: {entropy_stats.shape}, mean: {entropy_stats.mean():.3f}")

# Advanced reshaping patterns
print("\nAdvanced reshaping patterns:")

# Batch matrix multiplication setup
batch_matrices = np.random.randn(8, 32, 64)  # 8 matrices of 32x64
print(f"Batch matrices: {batch_matrices.shape}")

# Prepare for batched operations
transposed_batch = rearrange(batch_matrices, 'batch rows cols -> batch cols rows')
print(f"Transposed batch: {transposed_batch.shape}")

# Channel grouping for grouped convolutions
grouped_channels = np.random.randn(4, 64, 16, 16)  # batch, channels, height, width
groups = 8
channels_per_group = 64 // groups

grouped = rearrange(grouped_channels, 'b (groups cpg) h w -> b groups cpg h w', 
                   groups=groups, cpg=channels_per_group)
print(f"Grouped convolution: {grouped_channels.shape} -> {grouped.shape}")
print(f"  {groups} groups with {channels_per_group} channels each")

# Depthwise separable convolution preparation
depthwise_ready = rearrange(grouped_channels, 'b c h w -> b c 1 h w')
print(f"Depthwise conv ready: {grouped_channels.shape} -> {depthwise_ready.shape}")

print("\n✓ Multi-framework compatibility demonstrated successfully!")

In [ ]:
# Example 1: Basic Rearrange Operations and Tensor Reshaping
from einops import rearrange, reduce, repeat
import numpy as np

# Create sample data for demonstration
print("Basic rearrange operations and tensor reshaping:")
print("=" * 48)

# Create sample image batch: (batch, height, width, channels)
batch_size, height, width, channels = 4, 8, 8, 3
image_batch = np.random.randn(batch_size, height, width, channels)

print(f"Original image batch shape: {image_batch.shape}")
print(f"  Interpretation: {batch_size} images of {height}x{width} with {channels} channels")

# Example 1: Transpose operations
print(f"\n1. Basic transpose operations:")

# HWC to CHW (common in deep learning)
hwc_to_chw = rearrange(image_batch, 'batch height width channels -> batch channels height width')
print(f"   HWC to CHW: {image_batch.shape} -> {hwc_to_chw.shape}")

# Flatten spatial dimensions
flattened = rearrange(image_batch, 'batch height width channels -> batch (height width) channels')
print(f"   Flatten spatial: {image_batch.shape} -> {flattened.shape}")
print(f"     {height}*{width} = {height*width} spatial positions")

# Merge batch and spatial dimensions
merged = rearrange(image_batch, 'batch height width channels -> (batch height width) channels')
print(f"   Merge batch+spatial: {image_batch.shape} -> {merged.shape}")
print(f"     {batch_size}*{height}*{width} = {batch_size*height*width} pixels")

# Example 2: Patch extraction (common in Vision Transformers)
print(f"\n2. Patch extraction for Vision Transformers:")

# Create larger image for patch demonstration
large_image = np.random.randn(1, 32, 32, 3)  # 32x32 image
patch_size = 8

print(f"   Large image shape: {large_image.shape}")

# Extract non-overlapping patches
patches = rearrange(large_image, 
                   'batch (h patch_h) (w patch_w) channels -> batch (h w) (patch_h patch_w channels)',
                   patch_h=patch_size, patch_w=patch_size)

num_patches_h = 32 // patch_size
num_patches_w = 32 // patch_size
print(f"   Extracted patches: {patches.shape}")
print(f"     {num_patches_h}x{num_patches_w} = {num_patches_h*num_patches_w} patches")
print(f"     Each patch: {patch_size}x{patch_size}x{channels} = {patch_size*patch_size*channels} features")

# Restore patches back to image
restored = rearrange(patches,
                    'batch (h w) (patch_h patch_w channels) -> batch (h patch_h) (w patch_w) channels',
                    h=num_patches_h, w=num_patches_w, 
                    patch_h=patch_size, patch_w=patch_size, channels=channels)

print(f"   Restored image: {restored.shape}")
print(f"   Restoration successful: {np.allclose(large_image, restored)}")

# Example 3: Multi-head attention reshaping
print(f"\n3. Multi-head attention tensor reshaping:")

sequence_length, embed_dim, num_heads = 50, 768, 12
head_dim = embed_dim // num_heads

# Simulate query/key/value tensors
qkv = np.random.randn(1, sequence_length, embed_dim)  # (batch, seq_len, embed_dim)
print(f"   Input tensor: {qkv.shape}")

# Split into multiple heads
multi_head = rearrange(qkv, 'batch seq_len (num_heads head_dim) -> batch num_heads seq_len head_dim',
                      num_heads=num_heads, head_dim=head_dim)
print(f"   Multi-head format: {multi_head.shape}")
print(f"     {num_heads} heads, each with {head_dim} dimensions")

# Merge heads back
merged_heads = rearrange(multi_head, 'batch num_heads seq_len head_dim -> batch seq_len (num_heads head_dim)')
print(f"   Merged back: {merged_heads.shape}")
print(f"   Merge successful: {np.allclose(qkv, merged_heads)}")

print("\nBasic rearrange operations completed successfully!")

In [ ]:
# Example 2: Reduce Operations and Data Aggregation
from einops import rearrange, reduce, repeat
import numpy as np

print("Reduce operations and data aggregation:")
print("=" * 39)

# Create sample data for different reduction scenarios
batch_size, seq_len, features = 8, 100, 256
time_series = np.random.randn(batch_size, seq_len, features)

print(f"Time series data shape: {time_series.shape}")
print(f"  {batch_size} sequences, {seq_len} time steps, {features} features each")

# Example 1: Basic reductions along different axes
print(f"\n1. Basic reduction operations:")

# Temporal averaging (mean across time steps)
temporal_mean = reduce(time_series, 'batch time features -> batch features', 'mean')
print(f"   Temporal mean: {time_series.shape} -> {temporal_mean.shape}")
print(f"     Average across {seq_len} time steps")

# Feature averaging (mean across features)
feature_mean = reduce(time_series, 'batch time features -> batch time', 'mean')
print(f"   Feature mean: {time_series.shape} -> {feature_mean.shape}")
print(f"     Average across {features} features")

# Global average (mean across all dimensions except batch)
global_mean = reduce(time_series, 'batch time features -> batch', 'mean')
print(f"   Global mean: {time_series.shape} -> {global_mean.shape}")
print(f"     Single value per sequence")

# Example 2: Different reduction operations
print(f"\n2. Different reduction operations:")

# Create image data for spatial reductions
images = np.random.randn(4, 64, 64, 3)  # batch, height, width, channels
print(f"   Image data: {images.shape}")

# Max pooling (common in CNNs)
max_pooled = reduce(images, 'batch (h h2) (w w2) channels -> batch h w channels', 'max', h2=2, w2=2)
print(f"   Max pooling (2x2): {images.shape} -> {max_pooled.shape}")

# Global average pooling
global_avg_pool = reduce(images, 'batch height width channels -> batch channels', 'mean')
print(f"   Global avg pooling: {images.shape} -> {global_avg_pool.shape}")

# Min and max reductions
min_values = reduce(images, 'batch height width channels -> batch channels', 'min')
max_values = reduce(images, 'batch height width channels -> batch channels', 'max')
print(f"   Min values: {min_values.shape}, Max values: {max_values.shape}")

# Sum reduction with statistics
sum_values = reduce(images, 'batch height width channels -> batch channels', 'sum')
print(f"   Sum values: {sum_values.shape}")

# Example 3: Advanced reduction patterns
print(f"\n3. Advanced reduction patterns:")

# Create 3D convolution data: (batch, time, height, width, channels)
video_data = np.random.randn(2, 16, 32, 32, 3)  # 2 videos, 16 frames each
print(f"   Video data: {video_data.shape}")

# Spatial-temporal reductions
spatial_mean = reduce(video_data, 'batch time height width channels -> batch time channels', 'mean')
print(f"   Spatial averaging: {video_data.shape} -> {spatial_mean.shape}")
print(f"     Average across {32}x{32} spatial dimensions")

temporal_max = reduce(video_data, 'batch time height width channels -> batch height width channels', 'max')
print(f"   Temporal max: {video_data.shape} -> {temporal_max.shape}")
print(f"     Max across {16} time frames")

# Partial spatial reduction (reduce only height)
height_reduced = reduce(video_data, 'batch time height width channels -> batch time width channels', 'mean')
print(f"   Height reduction: {video_data.shape} -> {height_reduced.shape}")

# Example 4: Statistical reductions and combinations
print(f"\n4. Multiple reduction strategies:")

# Create sample batch for statistics
sample_data = np.random.randn(10, 50, 128)  # 10 samples, 50 time steps, 128 features
print(f"   Sample data: {sample_data.shape}")

# Compute various statistics
stats_results = {}
for operation in ['mean', 'max', 'min', 'sum']:
    result = reduce(sample_data, 'batch time features -> batch features', operation)
    stats_results[operation] = result
    print(f"   {operation.capitalize()}: {result.shape}, range: [{result.min():.3f}, {result.max():.3f}]")

# Combine reductions for normalization-like operations
# Standard deviation approximation using mean of squares
mean_vals = reduce(sample_data, 'batch time features -> batch features', 'mean')
squared_data = sample_data ** 2
mean_squared = reduce(squared_data, 'batch time features -> batch features', 'mean')
variance_approx = mean_squared - mean_vals ** 2

print(f"   Variance approximation shape: {variance_approx.shape}")
print(f"   Mean variance: {variance_approx.mean():.6f}")

# Example 5: Grouped reductions
print(f"\n5. Grouped and windowed reductions:")

# Create sequence data for windowed operations
sequence = np.random.randn(1, 120, 64)  # Long sequence
window_size = 10

print(f"   Long sequence: {sequence.shape}")

# Window-based averaging (group consecutive time steps)
windowed = rearrange(sequence[:, :120, :], 'batch (windows window_size) features -> batch windows window_size features', 
                    window_size=window_size)
windowed_mean = reduce(windowed, 'batch windows window_size features -> batch windows features', 'mean')

print(f"   Windowed data: {windowed.shape}")
print(f"   Windowed means: {windowed_mean.shape}")
print(f"     {120//window_size} windows of size {window_size}")

# Channel-wise reductions for feature maps
feature_maps = np.random.randn(4, 128, 32, 32)  # batch, channels, height, width
channel_stats = {}
for op in ['mean', 'max']:
    result = reduce(feature_maps, 'batch channels height width -> channels', op)
    channel_stats[op] = result
    print(f"   Channel-wise {op}: {result.shape}")

print("\nReduce operations and aggregation completed successfully!")

In [ ]:
# Example 3: Repeat Operations and Data Augmentation
from einops import rearrange, reduce, repeat
import numpy as np

print("Repeat operations and data augmentation:")
print("=" * 40)

# Create base data for repetition examples
base_vector = np.array([1, 2, 3, 4, 5])
base_matrix = np.random.randn(3, 4)
base_image = np.random.randn(1, 28, 28, 1)  # Single grayscale image

print(f"Base vector: {base_vector.shape} = {base_vector}")
print(f"Base matrix: {base_matrix.shape}")
print(f"Base image: {base_image.shape}")

# Example 1: Basic repetition patterns
print(f"\n1. Basic repetition patterns:")

# Repeat vector along new dimension
repeated_vector = repeat(base_vector, 'elements -> batch elements', batch=3)
print(f"   Vector repetition: {base_vector.shape} -> {repeated_vector.shape}")
print(f"     Created {3} copies along batch dimension")

# Repeat vector elements
element_repeated = repeat(base_vector, 'elements -> elements repeats', repeats=2)
print(f"   Element repetition: {base_vector.shape} -> {element_repeated.shape}")
print(f"     Each element repeated {2} times: {element_repeated}")

# Repeat matrix along multiple dimensions
repeated_matrix = repeat(base_matrix, 'height width -> batch height width channels', 
                        batch=2, channels=3)
print(f"   Matrix repetition: {base_matrix.shape} -> {repeated_matrix.shape}")
print(f"     {2} batches, {3} channels")

# Example 2: Data augmentation with repetition
print(f"\n2. Data augmentation with repetition:")

# Create batch from single image
image_batch = repeat(base_image, '1 height width channels -> batch height width channels', batch=8)
print(f"   Single to batch: {base_image.shape} -> {image_batch.shape}")

# Channel repetition (grayscale to RGB-like)
rgb_like = repeat(base_image, 'batch height width 1 -> batch height width channels', channels=3)
print(f"   Grayscale to RGB: {base_image.shape} -> {rgb_like.shape}")

# Spatial upsampling via repetition
upsampled = repeat(base_image, 'batch height width channels -> batch (height h_repeat) (width w_repeat) channels',
                  h_repeat=2, w_repeat=2)
print(f"   Spatial upsampling: {base_image.shape} -> {upsampled.shape}")
print(f"     Each pixel repeated in 2x2 blocks")

# Example 3: Pattern generation and tiling
print(f"\n3. Pattern generation and tiling:")

# Create a small pattern
pattern = np.array([[1, 0], [0, 1]])  # 2x2 checkerboard
print(f"   Pattern shape: {pattern.shape}")

# Tile pattern to create larger image
tiled_pattern = repeat(pattern, 'h w -> (tile_h h) (tile_w w)', tile_h=4, tile_w=4)
print(f"   Tiled pattern: {pattern.shape} -> {tiled_pattern.shape}")
print(f"     4x4 tiling creates {tiled_pattern.shape[0]}x{tiled_pattern.shape[1]} image")

# Create broadcasting-like behavior
row_vector = np.array([[1, 2, 3, 4]])  # 1x4
col_vector = np.array([[1], [2], [3]])  # 3x1

# Broadcast to match dimensions
broadcasted_row = repeat(row_vector, '1 width -> height width', height=3)
broadcasted_col = repeat(col_vector, 'height 1 -> height width', width=4)

print(f"   Row broadcasting: {row_vector.shape} -> {broadcasted_row.shape}")
print(f"   Col broadcasting: {col_vector.shape} -> {broadcasted_col.shape}")

# Element-wise product (like broadcasting)
broadcasted_product = broadcasted_row * broadcasted_col
print(f"   Broadcasted product shape: {broadcasted_product.shape}")

# Example 4: Sequence and temporal repetitions
print(f"\n4. Sequence and temporal repetitions:")

# Create a base sequence
sequence = np.array([1, 2, 3])
print(f"   Base sequence: {sequence}")

# Repeat sequence in time
temporal_repeat = repeat(sequence, 'seq -> time seq', time=5)
print(f"   Temporal repeat: {sequence.shape} -> {temporal_repeat.shape}")

# Create cyclic patterns
cyclic_sequence = repeat(sequence, 'seq -> (cycles seq)', cycles=3)
print(f"   Cyclic sequence: {sequence.shape} -> {cyclic_sequence.shape}")
print(f"     Result: {cyclic_sequence}")

# Time series with repeated patterns
base_signal = np.sin(np.linspace(0, 2*np.pi, 10))
repeated_signal = repeat(base_signal, 'samples -> batch samples repeats', batch=3, repeats=2)
print(f"   Signal repetition: {base_signal.shape} -> {repeated_signal.shape}")

# Example 5: Advanced repetition with combinations
print(f"\n5. Advanced repetition with rearrange combinations:")

# Create embedding-like repetitions
token_id = np.array([1])  # Single token
embedding_dim = 64

# Repeat token across sequence length and embedding dimensions
token_repeated = repeat(token_id, '1 -> seq_len embed_dim', seq_len=20, embed_dim=embedding_dim)
print(f"   Token embedding repeat: {token_id.shape} -> {token_repeated.shape}")

# Position encoding-like pattern
position_ids = np.arange(10)
position_repeated = repeat(position_ids, 'seq -> batch seq features', batch=4, features=32)
print(f"   Position repeat: {position_ids.shape} -> {position_repeated.shape}")

# Complex pattern: create attention mask
# Note: einops doesn't allow duplicate dimension names, so we need to build this in two steps
single_mask_value = np.array([1])  # Attention mask value
# First, expand to (batch, seq_len)
attention_mask_1d = repeat(single_mask_value, '1 -> batch seq_len', batch=2, seq_len=8)
# Then, expand the last dimension to create (batch, seq_len, seq_len)
attention_mask = repeat(attention_mask_1d, 'batch seq_len -> batch seq_len seq_len2', seq_len2=8)
print(f"   Attention mask: {single_mask_value.shape} -> {attention_mask.shape}")

# Combining repeat with rearrange for complex transformations
base_feature = np.random.randn(1, 8)
complex_transform = repeat(base_feature, '1 features -> batch (groups features)', 
                          batch=4, groups=3)
print(f"   Complex transform: {base_feature.shape} -> {complex_transform.shape}")

# Create data for kernel operations
kernel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]]) / 4  # Edge detection kernel
kernel_batch = repeat(kernel, 'h w -> batch channels h w', batch=1, channels=1)
print(f"   Kernel for convolution: {kernel.shape} -> {kernel_batch.shape}")

print("\nRepeat operations and data augmentation completed successfully!")